# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, referencing dataset structure via Croissant schema `@id` fields for full traceability.

### Dataset Source
The dataset source is described by its Croissant schema and available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the 'mlcroissant' library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset metadata and structured records using the [mlcroissant](https://github.com/mlcommons/croissant) library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset name and description
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Explore and enumerate all available record sets, their fields, and columns, referencing entities by their `@id`.

The following code lists all record sets, and for each, prints their `@id` plus the contained field and column `@id`s.

In [ ]:
# List all record sets and their details by @id
print("Record Sets and Fields (referenced by @id):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields and their columns:")
    for field in fields:
        fid = field.get('@id', str(field)) if isinstance(field, dict) else str(field)
        print(f"    Field @id: {fid}")
        if isinstance(field, dict) and 'column' in field:
            columns = field['column']
            if not isinstance(columns, list): columns = [columns]
            for col in columns:
                cid = col.get('@id', str(col)) if isinstance(col, dict) else str(col)
                print(f"      Column @id: {cid}")
    print('-'*60)

#### Example: Listing example records from the first record set
Now we print a sample of the records from the first available record set, using its `@id` (update the cell if you want other record sets).

In [ ]:
# Print out a sample record from the first record set
if record_sets:
    first_rs_id = record_sets[0]["@id"]
    print(f"Records for record set @id: {first_rs_id}\n")
    for idx, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if idx >= 2:  # Print only 3 sample records
            break

## 3. Data Extraction
Load all records for each record set into Pandas DataFrames, using their Croissant `@id`.

We capture all record sets by their `@id` (as strings), which ensures all references are explicit and traceable.

In [ ]:
# Extract all data into DataFrames (using record set @id's)
import collections

dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    # Retrieve all records from this record set
    records = list(dataset.records(record_set=rs_id))
    # Store as DataFrame
    dataframes[rs_id] = pd.DataFrame(records)
# Display columns from the first available record set
if record_set_ids:
    rs0 = record_set_ids[0]
    print(f"Columns in DataFrame for record set @id '{rs0}':")
    print(dataframes[rs0].columns.tolist())
    display(dataframes[rs0].head())

## 4. Exploratory Data Analysis (EDA)
Apply some preliminary data processing and transformations to one of the main record sets.

We'll demonstrate filtering, normalization, and group analysis based on specific fields referenced by `@id`.

Replace the variables below with actual `@id`s for the numeric and group fields of interest based on the printed columns from above.

In [ ]:
# Choose the main record set for analysis
main_rs_id = record_set_ids[0] if record_set_ids else None

if main_rs_id and not dataframes[main_rs_id].empty:
    df = dataframes[main_rs_id]

    # --- Specify the numeric field and group field by their @id string ---
    # Example: Use the first numeric column you have
    # Replace these accordingly with known @id's printed previously
    from pandas.api.types import is_numeric_dtype

    numeric_field_id = None
    group_field_id = None

    # Try to infer a numeric field from the dataframe's columns
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try to use a different column for grouping (categorical/string)
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '{main_rs_id}' with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the group field (if found)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').reset_index()
            print(f"\nGrouped filtered data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print(f"\nNo suitable group field found for grouping.")
    else:
        print("No numeric field found in DataFrame for EDA.")
else:
    print("No non-empty record set available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field, and optionally the grouped mean by a categorical field.

Update the field `@id`s as above to match your data for optimal presentation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id and (numeric_field_id in df.columns):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion

In this notebook, you have:
- Loaded FAIR² dataset metadata and records via the Croissant schema.
- Explored the structure—record sets and fields—using explicit `@id` references as per the Croissant specification.
- Extracted data into DataFrames, performed common EDA tasks (filtering, normalization, grouping), and visualized key distributions.

For deeper domain insights, refer to [dataset documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) or consult with clinical data experts as needed. You can iterate on this notebook by substituting field `@id`s for further targeted analysis.